# Vachan V2 — Per-Persona Tone Dial (Control Vectors)

**What this adds on top of the basic spike:** instead of a generic Hinglish↔English axis, the vector is built from a *specific persona's own anchors* — their real Hinglish phrases vs the English translations of those same phrases. The result is a **personalised** tone dial that captures *how that persona code-mixes*, not just "any Hinglish".

**Pre-requisite:** run `hinglish_control_vector_kaggle.ipynb` first and confirm it works. Same Kaggle setup (GPU T4, Internet On).

## 0. Persona Config — edit only this cell

Drop in any Vachan persona here. `hinglish_anchors` are real phrases *this persona actually says*. `english_anchors` are the direct English translations of the same meaning — same semantics, different tone. The vector will capture the *tone difference*, not a meaning difference.

In [ ]:
# ── EDIT THIS CELL FOR EACH PERSONA ──────────────────────────────────────────
# Persona name (used for saving the vector file)
PERSONA_NAME = "priya"

# Real Hinglish phrases this persona uses (positive end of the dial).
# 8-20 phrases gives a clean vector. More is better.
HINGLISH_ANCHORS = [
    "haan bhai, kal tak ho jayega — tension mat le",
    "yaar sach mein bahut mushkil tha, but we figured it out",
    "ek second ruk, main check karti hoon",
    "deployment ho gayi, sab theek hai ab",
    "acha woh wala issue? fix kar diya maine",
    "bata de kab chahiye, I'll make it happen",
    "thoda time lagega but pakka karungi",
    "okay so basically kya hua hai...",
    "testing chal rahi hai, ek ghante mein bolunga",
    "arre woh to simple hai, abhi karte hain",
    "koi baat nahi, kal fresh eyes se dekhte hain",
    "yeh wala approach better lagta hai mujhe",
]

# Direct English translations of the above (negative end — same meaning, formal tone).
# Keep the list the same length as HINGLISH_ANCHORS.
ENGLISH_ANCHORS = [
    "Yes, it will be done by tomorrow — no need to worry",
    "It was genuinely difficult, but we resolved it",
    "One moment, let me check",
    "The deployment is complete, everything is stable now",
    "That issue you mentioned? I have fixed it",
    "Let me know when you need it by and I will make it happen",
    "It will take some time but I will ensure it is done",
    "So to explain what happened here...",
    "Testing is in progress, I will update you within the hour",
    "That part is straightforward, we can do it right now",
    "No problem, we can revisit it tomorrow with fresh eyes",
    "This approach seems better to me",
]

assert len(HINGLISH_ANCHORS) == len(ENGLISH_ANCHORS), "Lists must be the same length"
print(f"Persona: {PERSONA_NAME} | {len(HINGLISH_ANCHORS)} anchor pairs loaded")

## 1. Install & Load Model

In [ ]:
!pip install -q repeng transformers accelerate bitsandbytes

In [ ]:
import torch, pickle, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"  # non-gated, no HF token needed

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 2. Build Contrastive Dataset from Persona Anchors

Each anchor pair becomes multiple `DatasetEntry` items by appending short suffixes. The suffix gives repeng more hidden-state positions to read — more positions → cleaner vector direction.

In [ ]:
SUFFIXES = [
    "", " and", " so", " but", " I", " We", " The", " It",
]

def anchor_to_prompt(text: str, suffix: str) -> str:
    """Wrap a raw anchor phrase in a chat template so repeng reads it the same way
    the model processes actual conversation turns."""
    msgs = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Send a quick update."},
        {"role": "assistant", "content": text},
    ]
    s = tokenizer.apply_chat_template(msgs, tokenize=False)
    return s + suffix

dataset = [
    DatasetEntry(
        positive=anchor_to_prompt(hi, sfx),
        negative=anchor_to_prompt(en, sfx),
    )
    for hi, en in zip(HINGLISH_ANCHORS, ENGLISH_ANCHORS)
    for sfx in SUFFIXES
]

print(f"{len(dataset)} contrastive pairs ({len(HINGLISH_ANCHORS)} anchors × {len(SUFFIXES)} suffixes)")
print("\n--- sample positive (tail 200 chars) ---")
print(dataset[0].positive[-200:])
print("\n--- sample negative (tail 200 chars) ---")
print(dataset[0].negative[-200:])

## 3. Train the Persona Vector

No gradient descent. repeng reads hidden states for each pair, subtracts them, and averages the difference direction across all pairs and layers. Takes ~1-2 min per 100 pairs on a T4.

In [ ]:
model.reset()
persona_vector = ControlVector.train(model, tokenizer, dataset)

layers = list(persona_vector.directions.keys())
print(f"vector trained over {len(layers)} layers")
print(f"direction shape per layer: {persona_vector.directions[layers[0]].shape}")

## 4. Save the Vector

Save as a `.pkl` file so you can reuse it across Kaggle sessions without retraining. Download it from the output panel and commit to `artifacts/control_vectors/` in the repo.

In [ ]:
OUT_PATH = f"/kaggle/working/{PERSONA_NAME}_tone_vector.pkl"
with open(OUT_PATH, "wb") as f:
    pickle.dump(persona_vector, f)
print(f"saved → {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1024:.1f} KB)")

## 5. Coeff Sweep — Dial Diagnostic

Test the full range. If the text shifts smoothly with the coeff, the vector is clean. If it degenerates above ~2.5, that's normal — stay in the ±1.5 to ±2.0 range for production.

In [ ]:
def generate(prompt: str, coeff: float, max_new_tokens: int = 90) -> str:
    model.reset()
    if coeff != 0.0:
        model.set_control(persona_vector, coeff)
    msgs = [{"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()

TEST_PROMPT = "Can you give me an update on the deployment?"

print(f"\nPersona: {PERSONA_NAME} | Prompt: {TEST_PROMPT!r}\n")
for coeff in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    label = f"coeff {coeff:+.1f}".ljust(12)
    print(f"  {label}  → {generate(TEST_PROMPT, coeff)}")

## 6. Fidelity Cosine Score (Stub)

This stub measures how *consistent* the persona's voice is across 3 outputs at the same coeff. It computes the average cosine similarity of their embedding centroids — matching the `av_cosine` metric already tracked in the Vachan Fidelity Ring. Replace the `embed()` function with your actual embedding call (Groq, OpenAI, or a local sentence-transformer) when you integrate.

In [ ]:
import numpy as np

# ── STUB: replace with your real embedding call ──
def embed(text: str) -> np.ndarray:
    """Fake 768-dim embedding for testing the pipeline without an API key.
    Swap this for: sentence_transformers, openai.Embedding, or Groq embed.
    """
    rng = np.random.default_rng(abs(hash(text[:40])) % (2**31))
    v = rng.standard_normal(768).astype(np.float32)
    return v / np.linalg.norm(v)
# ────────────────────────────────────────────────

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

TARGET_COEFF = 1.5  # the coeff you plan to use in production
N_SAMPLES = 3        # generate N outputs, measure how similar they are

prompts = [
    "Give me an update on the deployment.",
    "Is the new feature ready?",
    "What's the status of the bug fix?",
]

outputs = [generate(p, TARGET_COEFF) for p in prompts[:N_SAMPLES]]
embeds = [embed(o) for o in outputs]

centroid = np.mean(embeds, axis=0)
av_cosine = np.mean([cosine(e, centroid) for e in embeds])

print(f"Persona: {PERSONA_NAME} | coeff={TARGET_COEFF}")
print(f"av_cosine (voice consistency): {av_cosine:.3f}  [target ≥ 0.70]")
for i, (p, o) in enumerate(zip(prompts[:N_SAMPLES], outputs)):
    print(f"  [{i+1}] {p!r}")
    print(f"       → {o}")
    print()

## 7. Next Steps (do NOT do these in this notebook)

| Step | What | Where |
|------|------|-------|
| 7-A | Swap `embed()` stub with real sentence-transformer or Groq embed | cell 6 above |
| 7-B | Confirm `av_cosine ≥ 0.70` at coeff 1.5 for each persona | Fidelity Ring tracking |
| 7-C | Download `{persona_name}_tone_vector.pkl` and commit to `artifacts/control_vectors/` | repo |
| 7-D | vLLM serving spike — inject the vector at inference time via a `/generate` endpoint | new notebook |
| 7-E | Wire the coeff to the Fidelity Ring: if `av_cosine` drops below gate, raise coeff by 0.25 | Vachan backend |

**Tuning guide:**
- `av_cosine < 0.60` at coeff 1.0 → add more anchor pairs (aim for 20+)
- Text degenerates at coeff ≥ 2.5 → normal; production cap at 2.0
- Voice feels "too formal" even at +2 → widen layer band: `range(-3, -22, -1)`